In [1]:
!pip install torch torchvision pillow numpy pandas scikit-image tqdm

import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

from torchvision import models, transforms

from tqdm import tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5

In [2]:
import os, torch, random, numpy as np

if os.path.exists("/kaggle/input"):
    # Kaggle 環境
    BASE_DIR  = "/kaggle/input/hwk4-data"        # ← 資料集目錄名：hwk4-data
    DATA_ROOT = os.path.join(BASE_DIR, "hwk04_data")  # 這下面有 train/ test/ train.csv test.csv

    TRAIN_CSV_PATH = os.path.join(DATA_ROOT, "train.csv")
    TEST_CSV_PATH  = os.path.join(DATA_ROOT, "test.csv")
else:
    # 你自己電腦
    DATA_ROOT      = r"C:\研究所\深度學習\hwk04_data"
    TRAIN_CSV_PATH = os.path.join(DATA_ROOT, "train.csv")
    TEST_CSV_PATH  = os.path.join(DATA_ROOT, "test.csv")

print("DATA_ROOT     :", DATA_ROOT)
print("TRAIN_CSV_PATH:", TRAIN_CSV_PATH)
print("TEST_CSV_PATH :", TEST_CSV_PATH)

OUT_FCN_DIR  = "./FCN-8s"
OUT_UNET_DIR = "./U-Net"
os.makedirs(OUT_FCN_DIR, exist_ok=True)
os.makedirs(OUT_UNET_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 固定隨機種子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


DATA_ROOT     : /kaggle/input/hwk4-data/hwk04_data
TRAIN_CSV_PATH: /kaggle/input/hwk4-data/hwk04_data/train.csv
TEST_CSV_PATH : /kaggle/input/hwk4-data/hwk04_data/test.csv
Using device: cuda


In [3]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df  = pd.read_csv(TEST_CSV_PATH)

print("Train head:")
display(train_df.head())

print("Test head:")
display(test_df.head())


Train head:


,pre,post
0,/SonoMPEG/train/pre/frame1.bmp,/SonoMPEG/train/post/frame1.bmp_ROI.bmp
1,/SonoMPEG/train/pre/frame2.bmp,/SonoMPEG/train/post/frame2.bmp_ROI.bmp
2,/SonoMPEG/train/pre/frame3.bmp,/SonoMPEG/train/post/frame3.bmp_ROI.bmp
3,/SonoMPEG/train/pre/frame4.bmp,/SonoMPEG/train/post/frame4.bmp_ROI.bmp
4,/SonoMPEG/train/pre/frame5.bmp,/SonoMPEG/train/post/frame5.bmp_ROI.bmp


Test head:


,pre
0,/SonoMPEG/test/pre/frame301.bmp
1,/SonoMPEG/test/pre/frame302.bmp
2,/SonoMPEG/test/pre/frame303.bmp
3,/SonoMPEG/test/pre/frame304.bmp
4,/SonoMPEG/test/pre/frame305.bmp


In [4]:
def resolve_path_from_csv(root, csv_path_str):
 
    p = csv_path_str.replace("\\", "/")   # 保險用，全部換成 '/'
    if "train/" in p:
        sub = p[p.index("train/"):]       # 從 'train/...' 開始
    elif "test/" in p:
        sub = p[p.index("test/"):]        # 或 'test/...'
    else:
        sub = p.lstrip("/\\")             # 萬一沒有就單純去掉前面的 /
    return os.path.join(root, sub)


In [5]:
from torchvision import transforms

RESIZE_HW = (512, 512)  # 統一尺寸，避免 UNet 尺寸 mismatch

resize_tf = transforms.Resize(RESIZE_HW)

class CarotidTrainDataset(Dataset):
    def __init__(self, df, data_root):
        self.df = df
        self.data_root = data_root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        pre_path = resolve_path_from_csv(self.data_root, row["pre"])
        post_path = resolve_path_from_csv(self.data_root, row["post"])

        img = Image.open(pre_path).convert("RGB")
        mask = Image.open(post_path).convert("L")

        # **🔥 關鍵：統一 resize**
        img = resize_tf(img)
        mask = resize_tf(mask)

        # To tensor
        img = np.array(img).astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2, 0, 1)

        mask = np.array(mask).astype(np.float32) / 255.0
        mask = torch.from_numpy(mask)[None, :, :]

        return img, mask


In [6]:
import torch.nn.functional as F

def dice_loss_from_logits(logits, targets, eps=1e-7):
    """
    logits: (N,1,H,W) 未 sigmoid
    targets: (N,1,H,W) 0/1
    """
    probs = torch.sigmoid(logits)
    num = (probs * targets).sum(dim=(1,2,3))
    den = probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3)) + eps
    dice = 1 - (2 * num / den)
    return dice.mean()

bce_loss_fn = nn.BCEWithLogitsLoss()

def mixed_loss(logits, targets, alpha=0.7):
    """
    0.7 * Dice + 0.3 * BCE
    """
    d = dice_loss_from_logits(logits, targets)
    b = bce_loss_fn(logits, targets)
    return alpha * d + (1 - alpha) * b


In [7]:
class CarotidTestDataset(Dataset):
    def __init__(self, df, data_root):
        self.df = df
        self.data_root = data_root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pre_path = resolve_path_from_csv(self.data_root, row["pre"])

        img = Image.open(pre_path).convert("RGB")

        orig_size = img.size  # for restoring ROI size
        img_resized = resize_tf(img)

        img_arr = np.array(img_resized).astype(np.float32) / 255.0
        img_tensor = torch.from_numpy(img_arr).permute(2,0,1)

        return img_tensor, row["pre"].split("/")[-1], orig_size


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v2

class MobileSegNet(nn.Module):
    """
    MobileNetV2 當 encoder，做 FCN-8s：
    - 1/32: 最後一層 features 輸出
    - 1/16: features[13] 的輸出 (channels = 96)
    - 1/8 : features[4]  的輸出 (channels = 32)
    """
    def __init__(self, n_classes=1):
        super(MobileSegNet, self).__init__()

        backbone = mobilenet_v2(weights=None)
        self.features = backbone.features

        # 最後一層 conv 輸出 channels（1/32）
        last_c = backbone.last_channel   # 通常是 1280

        # 這三個 in_channels 要跟對應層的 channel 數一致
        self.score32 = nn.Conv2d(last_c,  n_classes, kernel_size=1)   # 1/32
        self.score16 = nn.Conv2d(96,      n_classes, kernel_size=1)   # 1/16
        self.score8  = nn.Conv2d(32,      n_classes, kernel_size=1)   # 1/8

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]

        h = x
        feat8  = None  # 1/8
        feat16 = None  # 1/16

        for i, blk in enumerate(self.features):
            h = blk(h)

            # ↓ 這兩個 index 是對應 MobileNetV2 的下採樣點
            if i == 4:      # 解析度約 1/8, channels = 32
                feat8 = h
            elif i == 13:   # 解析度約 1/16, channels = 96
                feat16 = h
            # 迴圈最後一層輸出就是 1/32

        feat32 = h  # 1/32, channels = last_c

        # ---------- FCN-8s：32 → 16 → 8 → full ---------- #

        # 1/32 → 1/16
        s32 = self.score32(feat32)  # (N, C, H/32, W/32)
        s32_up = F.interpolate(
            s32, size=feat16.shape[2:], mode="bilinear", align_corners=False
        )

        s16 = self.score16(feat16)  # (N, C, H/16, W/16)
        s16 = s16 + s32_up          # skip connection (FCN-16s)

        # 1/16 → 1/8
        s16_up = F.interpolate(
            s16, size=feat8.shape[2:], mode="bilinear", align_corners=False
        )

        s8 = self.score8(feat8)     # (N, C, H/8, W/8)
        s8 = s8 + s16_up            # skip connection (FCN-8s)

        # 1/8 → full resolution
        out = F.interpolate(
            s8, size=(H, W), mode="bilinear", align_corners=False
        )

        return out


In [17]:
class DoubleConvLite(nn.Module):
    """只做一層 3x3 conv + BN + ReLU，減少參數量"""
    def __init__(self, in_ch, out_ch):
        super(DoubleConvLite, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNetSmall(nn.Module):
    """
    精簡版 U-Net：
    - 3 層 encoder + 1 個 bottleneck + 3 層 decoder
    - 每層只用一個 conv block
    - 預設 base_ch=20 時，參數量約 37 萬
    """
    def __init__(self, n_classes=1, base_ch=32):
        super(UNetSmall, self).__init__()

        # Encoder
        self.enc1 = DoubleConvLite(3, base_ch)          # 3 -> 20
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConvLite(base_ch, base_ch * 2)  # 20 -> 40
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConvLite(base_ch * 2, base_ch * 4)  # 40 -> 80
        self.pool3 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConvLite(base_ch * 4, base_ch * 8)  # 80 -> 160

        # Decoder
        self.up3 = nn.ConvTranspose2d(base_ch * 8, base_ch * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConvLite(base_ch * 8, base_ch * 4)        # concat 80+80 -> 80

        self.up2 = nn.ConvTranspose2d(base_ch * 4, base_ch * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConvLite(base_ch * 4, base_ch * 2)        # concat 40+40 -> 40

        self.up1 = nn.ConvTranspose2d(base_ch * 2, base_ch, kernel_size=2, stride=2)
        self.dec1 = DoubleConvLite(base_ch * 2, base_ch)            # concat 20+20 -> 20

        self.out_conv = nn.Conv2d(base_ch, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        c1 = self.enc1(x)
        p1 = self.pool1(c1)

        c2 = self.enc2(p1)
        p2 = self.pool2(c2)

        c3 = self.enc3(p2)
        p3 = self.pool3(c3)

        # Bottleneck
        bn = self.bottleneck(p3)

        # Decoder
        u3 = self.up3(bn)
        c3_crop = center_crop_like(c3, u3)
        u3 = torch.cat([u3, c3_crop], dim=1)
        c3d = self.dec3(u3)

        u2 = self.up2(c3d)
        c2_crop = center_crop_like(c2, u2)
        u2 = torch.cat([u2, c2_crop], dim=1)
        c2d = self.dec2(u2)

        u1 = self.up1(c2d)
        c1_crop = center_crop_like(c1, u1)
        u1 = torch.cat([u1, c1_crop], dim=1)
        c1d = self.dec1(u1)

        out = self.out_conv(c1d)
        return out


In [10]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_dice = 0.0

    for imgs, masks in loader:   # ★ 只拿兩個
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        logits = model(imgs)

        loss = mixed_loss(logits, masks)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            dice = 1 - dice_loss_from_logits(logits, masks)

        total_loss += loss.item()
        total_dice += dice.item()

    return total_loss / len(loader), total_dice / len(loader)


In [11]:
def eval_one_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0

    with torch.no_grad():
        for imgs, masks in loader:   # ★ 一樣只有兩個
            imgs = imgs.to(device)
            masks = masks.to(device)

            logits = model(imgs)

            loss = mixed_loss(logits, masks)
            dice = 1 - dice_loss_from_logits(logits, masks)

            total_loss += loss.item()
            total_dice += dice.item()

    return total_loss / len(loader), total_dice / len(loader)


In [15]:
from sklearn.model_selection import KFold

# ---------- K-FOLD 設定 ---------- #
N_FOLDS = 3          # 你可以改成 4 或 3，看時間
EPOCHS_FCN = 80      # 沿用你原本的 epoch 數
BATCH_SIZE = 4       # 用你原本的 batch_size
PATIENCE = 10        # early stopping

print("Total train samples:", len(train_df))

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = []
global_best_dice = 0.0
global_best_path = None

for fold, (train_idx, val_idx) in enumerate(kf.split(train_df), 1):
    print(f"\n========== Fold {fold}/{N_FOLDS} ==========")
    print("Train idx size:", len(train_idx), " | Val idx size:", len(val_idx))

    # 這裡把 train_df 切成本 fold 的 train / val
    train_sub_df = train_df.iloc[train_idx].reset_index(drop=True)
    val_sub_df   = train_df.iloc[val_idx].reset_index(drop=True)

    # Dataset & DataLoader
    train_dataset = CarotidTrainDataset(train_sub_df, DATA_ROOT)
    val_dataset   = CarotidTrainDataset(val_sub_df,   DATA_ROOT)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader   = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # ---------- 每個 fold 都重新建一個新模型 (MobileNet) ---------- #
    fcn_model = MobileSegNet(n_classes=1).to(device)

    optimizer_fcn = torch.optim.Adam(
        fcn_model.parameters(),
        lr=1e-3   # 用你原本的 lr，先不要改
    )

    # Cosine LR (跟你 notebook 原本的一樣)
    scheduler_fcn = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_fcn,
        T_max=EPOCHS_FCN
    )

    best_val_dice_fcn = 0.0
    best_fold_path = f"best_mobilenetseg_fold{fold}.pth"
    no_improve = 0

    # ---------- 訓練這個 fold ---------- #
    for epoch in range(1, EPOCHS_FCN + 1):
        train_loss, train_dice = train_one_epoch(fcn_model, train_loader, optimizer_fcn)
        val_loss, val_dice     = eval_one_epoch(fcn_model, val_loader)

        scheduler_fcn.step()

        print(
            f"[Fold {fold}] Epoch {epoch}/{EPOCHS_FCN} "
            f"TrainLoss={train_loss:.4f} TrainDice={train_dice:.4f} "
            f"ValLoss={val_loss:.4f} ValDice={val_dice:.4f}"
        )

        # 這個 fold 的 best model
        if val_dice > best_val_dice_fcn:
            best_val_dice_fcn = val_dice
            no_improve = 0
            torch.save(fcn_model.state_dict(), best_fold_path)
            print(f"  -> New BEST for Fold {fold} (val dice={val_dice:.4f}), model saved.")
        else:
            no_improve += 1
            print(f"  -> No improvement for {no_improve} epoch(s).")

        if no_improve >= PATIENCE:
            print(f"Early stopping on Fold {fold}.")
            break

    fold_results.append((fold, best_val_dice_fcn, best_fold_path))

    # 同時記錄「全體最好的那個 fold」
    if best_val_dice_fcn > global_best_dice:
        global_best_dice = best_val_dice_fcn
        global_best_path = best_fold_path

print("\n=========== K-FOLD SUMMARY ===========")
for fold, dice, path in fold_results:
    print(f"Fold {fold}: best val dice = {dice:.4f}, ckpt = {path}")
print("--------------------------------------")
print(f"Best overall model: {global_best_path} (val dice={global_best_dice:.4f})")


Total train samples: 300

========== Fold 1/3 ==========
Train idx size: 200  | Val idx size: 100
[Fold 1] Epoch 1/80 TrainLoss=0.2119 TrainDice=0.7630 ValLoss=1.0931 ValDice=0.0002
  -> New BEST for Fold 1 (val dice=0.0002), model saved.
[Fold 1] Epoch 2/80 TrainLoss=0.0532 TrainDice=0.9388 ValLoss=0.0452 ValDice=0.9482
  -> New BEST for Fold 1 (val dice=0.9482), model saved.
[Fold 1] Epoch 3/80 TrainLoss=0.0374 TrainDice=0.9573 ValLoss=0.0404 ValDice=0.9550
  -> New BEST for Fold 1 (val dice=0.9550), model saved.
[Fold 1] Epoch 4/80 TrainLoss=0.0303 TrainDice=0.9654 ValLoss=0.0340 ValDice=0.9621
  -> New BEST for Fold 1 (val dice=0.9621), model saved.
[Fold 1] Epoch 5/80 TrainLoss=0.0262 TrainDice=0.9699 ValLoss=0.0313 ValDice=0.9651
  -> New BEST for Fold 1 (val dice=0.9651), model saved.
[Fold 1] Epoch 6/80 TrainLoss=0.0237 TrainDice=0.9729 ValLoss=0.0297 ValDice=0.9671
  -> New BEST for Fold 1 (val dice=0.9671), model saved.
[Fold 1] Epoch 7/80 TrainLoss=0.0211 TrainDice=0.9757 Va

In [18]:
from sklearn.model_selection import KFold

# ========= U-Net 5-FOLD K-FOLD TRAINING ========= #

N_FOLDS_UNET = 3          
EPOCHS_UNET  = 80
BATCH_SIZE   = 4
PATIENCE_UNET = 8

print("Total train samples:", len(train_df))

kf_unet = KFold(n_splits=N_FOLDS_UNET, shuffle=True, random_state=42)

unet_fold_results = []
unet_global_best_dice = 0.0
unet_global_best_path = None

for fold, (train_idx, val_idx) in enumerate(kf_unet.split(train_df), 1):
    print(f"\n========== [U-Net] Fold {fold}/{N_FOLDS_UNET} ==========")
    print("Train idx size:", len(train_idx), " | Val idx size:", len(val_idx))

    # 這裡把 train_df 切成本 fold 的 train / val
    train_sub_df = train_df.iloc[train_idx].reset_index(drop=True)
    val_sub_df   = train_df.iloc[val_idx].reset_index(drop=True)

    # Dataset & DataLoader
    train_dataset_unet = CarotidTrainDataset(train_sub_df, DATA_ROOT)
    val_dataset_unet   = CarotidTrainDataset(val_sub_df,   DATA_ROOT)

    train_loader_unet = DataLoader(
        train_dataset_unet,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader_unet = DataLoader(
        val_dataset_unet,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # ---------- 每個 fold 都重新建一個 U-Net 模型 ---------- #
    # 如果你有精簡版 U-Net，就在這裡換成 UNetSmall(n_classes=1, base_ch=20)
    unet_model = UNetSmall(n_classes=1, base_ch=32).to(device)

    optimizer_unet = torch.optim.Adam(unet_model.parameters(), lr=1e-3)
    scheduler_unet = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_unet, T_max=EPOCHS_UNET
    )

    best_val_dice_unet = 0.0
    best_unet_path = f"best_unet_fold{fold}.pth"
    no_improve = 0

    for epoch in range(1, EPOCHS_UNET + 1):
        train_loss, train_dice = train_one_epoch(unet_model, train_loader_unet, optimizer_unet)
        val_loss, val_dice     = eval_one_epoch(unet_model, val_loader_unet)

        scheduler_unet.step()

        print(f"[U-Net][Fold {fold}] Epoch {epoch}/{EPOCHS_UNET} "
              f"TrainLoss={train_loss:.4f} TrainDice={train_dice:.4f} "
              f"ValLoss={val_loss:.4f} ValDice={val_dice:.4f}")

        if val_dice > best_val_dice_unet:
            best_val_dice_unet = val_dice
            no_improve = 0
            torch.save(unet_model.state_dict(), best_unet_path)
            print(f"  -> New best U-Net on Fold {fold} (val dice={val_dice:.4f}), model saved.")
        else:
            no_improve += 1
            print(f"  -> No improvement for {no_improve} epoch(s).")

        if no_improve >= PATIENCE_UNET:
            print("Early stopping U-Net on this fold.")
            break

    unet_fold_results.append((fold, best_val_dice_unet, best_unet_path))

    if best_val_dice_unet > unet_global_best_dice:
        unet_global_best_dice = best_val_dice_unet
        unet_global_best_path = best_unet_path

print("\n=========== U-Net 5-FOLD SUMMARY ===========")
for fold, dice, path in unet_fold_results:
    print(f"Fold {fold}: best val dice = {dice:.4f}, ckpt = {path}")
print("--------------------------------------")
print(f"Best overall U-Net model: {unet_global_best_path} (val dice={unet_global_best_dice:.4f})")


Total train samples: 300

========== [U-Net] Fold 1/3 ==========
Train idx size: 200  | Val idx size: 100
[U-Net][Fold 1] Epoch 1/80 TrainLoss=0.6235 TrainDice=0.3286 ValLoss=0.7970 ValDice=0.3318
  -> New best U-Net on Fold 1 (val dice=0.3318), model saved.
[U-Net][Fold 1] Epoch 2/80 TrainLoss=0.4702 TrainDice=0.4687 ValLoss=0.4928 ValDice=0.4803
  -> New best U-Net on Fold 1 (val dice=0.4803), model saved.
[U-Net][Fold 1] Epoch 3/80 TrainLoss=0.3624 TrainDice=0.5827 ValLoss=0.5143 ValDice=0.4064
  -> No improvement for 1 epoch(s).
[U-Net][Fold 1] Epoch 4/80 TrainLoss=0.2942 TrainDice=0.6632 ValLoss=0.4411 ValDice=0.4964
  -> New best U-Net on Fold 1 (val dice=0.4964), model saved.
[U-Net][Fold 1] Epoch 5/80 TrainLoss=0.2718 TrainDice=0.6946 ValLoss=0.3291 ValDice=0.6454
  -> New best U-Net on Fold 1 (val dice=0.6454), model saved.
[U-Net][Fold 1] Epoch 6/80 TrainLoss=0.2578 TrainDice=0.7119 ValLoss=0.3091 ValDice=0.6781
  -> New best U-Net on Fold 1 (val dice=0.6781), model saved.
[U

In [19]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
from torch.utils.data import DataLoader
import torch

# --------------------------------------------------
# 1. 準備 Test DataFrame / Dataset / DataLoader
# --------------------------------------------------
DATA_ROOT = "/kaggle/input/hwk4-data/hwk04_data"   # 跟前面定義的一樣
TEST_CSV_PATH = os.path.join(DATA_ROOT, "test.csv")

test_df = pd.read_csv(TEST_CSV_PATH)
test_dataset = CarotidTestDataset(test_df, DATA_ROOT)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test size:", len(test_dataset))

# --------------------------------------------------
# 2. 讀入 K-FOLD 的 3 個 model 做 ensemble
# --------------------------------------------------
ckpt_paths = [
    "best_mobilenetseg_fold1.pth",
    "best_mobilenetseg_fold2.pth",
    "best_mobilenetseg_fold3.pth",
]

models = []
for p in ckpt_paths:
    if not os.path.exists(p):
        print(f"[WARNING] checkpoint not found: {p}")
        continue
    m = MobileSegNet(n_classes=1).to(device)
    m.load_state_dict(torch.load(p, map_location=device))
    m.eval()
    models.append(m)
    print(f"Loaded checkpoint: {p}")

assert len(models) > 0, "沒有任何 fold 的權重被載入，請確認 ckpt 檔名存在。"

print(f"Ensemble models: {len(models)} folds")

# --------------------------------------------------
# 3. Ensemble 推論並存成 .bmp_ROI.bmp
# --------------------------------------------------
PRED_DIR = "./FCN-8s"     # 讓後處理那一格可以沿用同一個變數
os.makedirs(PRED_DIR, exist_ok=True)
print("Saving prediction masks to:", PRED_DIR)

with torch.inference_mode():
    for imgs, file_name, orig_size in tqdm(test_loader, desc="MobileSegNet Ensemble Test"):
        # imgs: (1, 3, H, W)
        imgs = imgs.to(device)

        # ---- ensemble：平均各 fold 的 logits ----
        logits_sum = None
        for m in models:
            out = m(imgs)          # (1,1,H,W)
            if logits_sum is None:
                logits_sum = out
            else:
                logits_sum += out
        logits = logits_sum / len(models)

        # 轉成機率再 threshold
        probs = torch.sigmoid(logits)
        pred_mask = (probs > 0.4).float()   # 有需要可以改 0.45 / 0.55 微調

        # 轉成 0/255 mask
        mask_np = pred_mask.squeeze().cpu().numpy() * 255.0
        mask_np = mask_np.astype(np.uint8)
        mask_img = Image.fromarray(mask_np)

        # ----------------- 還原成原始大小 -----------------
        # CarotidTestDataset 回傳的 orig_size 是 (W, H) or (H, W)
        if isinstance(orig_size, (list, tuple)) and len(orig_size) == 2:
            ow, oh = orig_size
        else:
            ow = int(orig_size[0].item())
            oh = int(orig_size[1].item())
        # ------------------------------------------

        if mask_img.size != (ow, oh):
            mask_img = mask_img.resize((ow, oh), resample=Image.NEAREST)

        # frame301.bmp -> frame301.bmp_ROI.bmp
        fname = file_name[0] if isinstance(file_name, (list, tuple)) else file_name
        out_name = fname.replace(".bmp", ".bmp_ROI.bmp")
        out_path = os.path.join(PRED_DIR, out_name)
        mask_img.save(out_path)

print("Ensemble inference finished.")


Test size: 100
Loaded checkpoint: best_mobilenetseg_fold1.pth
Loaded checkpoint: best_mobilenetseg_fold2.pth
Loaded checkpoint: best_mobilenetseg_fold3.pth
Ensemble models: 3 folds
Saving prediction masks to: ./FCN-8s


MobileSegNet Ensemble Test:   0%|          | 0/100 [00:00<?, ?it/s]

Ensemble inference finished.


In [22]:
# ========= U-Net 3-FOLD ENSEMBLE INFERENCE ========= #

# 1. 讀入 3 個 U-Net fold 的權重
unet_ckpt_paths = [
    "best_unet_fold1.pth",
    "best_unet_fold2.pth",
    "best_unet_fold3.pth",
]

unet_models = []
for p in unet_ckpt_paths:
    if not os.path.exists(p):
        print(f"[WARNING] U-Net checkpoint not found: {p}")
        continue
    m = UNetSmall(n_classes=1, base_ch=32).to(device)
    m.load_state_dict(torch.load(p, map_location=device))
    m.eval()
    unet_models.append(m)
    print(f"Loaded U-Net checkpoint: {p}")

assert len(unet_models) > 0, "沒有載入到任何 U-Net 權重，請確認 .pth 檔案存在。"

print(f"U-Net ensemble models: {len(unet_models)} folds")

# 2. 在 test set 上做 ensemble 推論
with torch.inference_mode():
    for imgs, file_name, orig_size in tqdm(test_loader, desc="U-Net 3-fold Ensemble Test"):
        imgs = imgs.to(device)

        # 5 folds 的 logits 平均
        logits_sum = None
        for m in unet_models:
            out = m(imgs)  # (1,1,H,W)
            if logits_sum is None:
                logits_sum = out
            else:
                logits_sum += out
        logits = logits_sum / len(unet_models)

        # 轉機率 + 二值化
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()  # threshold 看你要不要之後再調

        # 取出單張 mask，轉成 0/255
        pred_mask = preds[0, 0].cpu().numpy().astype(np.uint8) * 255
        mask_img = Image.fromarray(pred_mask, mode="L")

        # 還原成原始大小
        if isinstance(orig_size, (list, tuple)):
            ow, oh = orig_size
        else:
            # 如果 CarotidTestDataset 給的是 tensor
            ow = int(orig_size[0].item())
            oh = int(orig_size[1].item())
        if mask_img.size != (ow, oh):
            mask_img = mask_img.resize((ow, oh), resample=Image.NEAREST)

        # 檔名處理
        if isinstance(file_name, (list, tuple)):
            fname = file_name[0]
        else:
            fname = file_name

        out_name = fname.replace(".bmp", ".bmp_ROI.bmp")
        out_path = os.path.join(OUT_UNET_DIR, out_name)
        mask_img.save(out_path)

print("U-Net 3-fold ensemble inference finished.")



Loaded U-Net checkpoint: best_unet_fold1.pth
Loaded U-Net checkpoint: best_unet_fold2.pth
Loaded U-Net checkpoint: best_unet_fold3.pth
U-Net ensemble models: 3 folds


U-Net 3-fold Ensemble Test:   0%|          | 0/100 [00:00<?, ?it/s]

/tmp/ipykernel_47/3103031566.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  mask_img = Image.fromarray(pred_mask, mode="L")


U-Net 3-fold ensemble inference finished.


In [23]:
import zipfile

def zip_dir(src_dir, zip_name):
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(src_dir):
            for f in files:
                full_path = os.path.join(root, f)
                rel_path = os.path.relpath(full_path, src_dir)
                zf.write(full_path, arcname=rel_path)

zip_dir(OUT_FCN_DIR,  "FCN-8s.zip")
zip_dir(OUT_UNET_DIR, "U-Net.zip")

print("Saved FCN-8s.zip and U-Net.zip")



Saved FCN-8s.zip and U-Net.zip


In [24]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

fcn_params  = count_params(MobileSegNet(n_classes=1))
unet_params = count_params(UNet(n_classes=1, base_ch=64))

print("MobileSegNet params:", fcn_params)
print("U-Net params:", unet_params)


MobileSegNet params: 2225283
U-Net params: 31043521


In [25]:
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import os

PRED_DIR = "./FCN-8s"
DATA_ROOT = "/kaggle/input/hwk4-data/hwk04_data"
TEST_CSV_PATH = os.path.join(DATA_ROOT, "test.csv")

def rle_encode(mask):
    """
    mask: 0/1 numpy array, shape (H,W)
    以 Kaggle 要的方式做 RLE，長度為 H*W (column-major, order='F')
    """
    pixels = mask.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

def postprocess_mask(mask_bin, kernel_size=5, min_area=300):
    """
    mask_bin: 0/1 numpy array
    先做 morphological opening，再移除小區塊
    """
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    opened = cv2.morphologyEx(mask_bin, cv2.MORPH_OPEN, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(opened, connectivity=8)
    out = np.zeros_like(opened)
    for i in range(1, num_labels):  # 0 是 background
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_area:
            out[labels == i] = 1
    return out

test_df = pd.read_csv(TEST_CSV_PATH)
records = []

print("Using prediction dir:", PRED_DIR)
print("TEST_CSV_PATH:", TEST_CSV_PATH)

for _, row in test_df.iterrows():
    full_path = row["pre"]              # "/SonoMPEG/test/pre/frame301.bmp"
    fname = os.path.basename(full_path) # "frame301.bmp"

    roi_name = fname.replace(".bmp", ".bmp_ROI.bmp")
    roi_path = os.path.join(PRED_DIR, roi_name)

    # 讀取預測 mask 圖 (0~255)
    img = Image.open(roi_path).convert("L")
    mask = np.array(img)

    # 二值化 ➜ 後處理 ➜ RLE
    mask_bin = (mask > 127).astype("uint8")
    mask_pp  = postprocess_mask(mask_bin, kernel_size=5, min_area=300)
    rle      = rle_encode(mask_pp)

    records.append({
        "pre": fname,
        "rle_encode": rle
    })

submission = pd.DataFrame(records)
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
display(submission.head())
print("Rows:", len(submission))


Using prediction dir: ./FCN-8s
TEST_CSV_PATH: /kaggle/input/hwk4-data/hwk04_data/test.csv
Saved submission.csv


,pre,rle_encode
0,frame301.bmp,11060 17 11116 25 11622 87 12185 91 12646 27 1...
1,frame302.bmp,11622 20 11658 43 12177 92 12738 100 13304 100...
2,frame303.bmp,11059 14 11124 12 11622 84 12185 88 12749 91 1...
3,frame304.bmp,11111 18 11614 84 12173 92 12738 94 13304 94 1...
4,frame305.bmp,11108 20 11608 89 12172 92 12737 94 13303 94 1...


Rows: 100
